In [1]:
import os
import django

# Replace 'Hobart.settings' with the actual path to your settings file
# Based on your file path, it's likely 'Hobart.settings'
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'Hobart.settings')

# This line "boots up" Django
django.setup()

In [4]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
import json

BASE = "https://www.hobart.ca"
START_MENU = f"{BASE}/?lang=fr"

visited = set()
results = []


== Category: Boulangerie ==
Scraping https://www.hobart.ca/boulangerie/?lang=fr
  Scraping https://www.hobart.ca/boulangerie/?lang=fr#content
    Scraping https://www.hobart.ca/boulangerie/fours/?lang=fr
      Scraping https://www.hobart.ca/boulangerie/fours/?lang=fr#content
        Scraping https://www.hobart.ca/boulangerie/fours/four-a-chariot-rotatif-cs500/?lang=fr
          Scraping https://www.hobart.ca/boulangerie/fours/four-a-chariot-rotatif-cs500/?lang=fr#content
            Scraping https://www.hobart.ca/boulangerie/fours/four-a-chariot/?lang=fr
              Scraping https://www.hobart.ca/boulangerie/fours/four-a-chariot/?lang=fr#content
                Scraping https://www.hobart.ca/boulangerie/fours/mini-rotating-rack-oven/?lang=fr
                  Scraping https://www.hobart.ca/boulangerie/fours/mini-rotating-rack-oven/?lang=fr#content
                    Scraping https://www.hobart.ca/boulangerie/etuves-de-fermentation/?lang=fr
                      Scraping https://www

In [6]:
import re
import json

INPUT_FILE = "DL/Hobart/hyperlinks hobart canada.rtf"
OUTPUT_TREE = "hobart_tree.json"
OUTPUT_LEAVES = "hobart_leaf_urls.json"

def extract_lines_from_rtf(path):
    """Extracts text lines containing 'Scraping' or '== Category:' from the RTF file."""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    # Strip RTF control words and braces
    text = re.sub(r"\\[a-z]+\d*|[{}]", "", text)
    text = re.sub(r"[\r\n]+", "\n", text)

    lines = []
    for line in text.split("\n"):
        line = line.strip()
        if not line:
            continue
        if line.startswith("== Category:") or "Scraping" in line:
            lines.append(line)
    return lines

def build_adjacency(lines):
    """Builds a tree (adjacency list) based on indentation."""
    adjacency = {}
    stack = []  # will hold tuples (depth, url)
    current_category = None

    for line in lines:
        if line.startswith("== Category:"):
            current_category = line.replace("== Category:", "").strip()
            continue

        # Extract depth and URL
        m = re.match(r"(\s*)Scraping\s+(.*)", line)
        if not m:
            continue
        depth = len(m.group(1)) // 2  # each 2 spaces = 1 depth
        url = m.group(2).strip()

        # Clean up URL
        url = re.sub(r"#.*", "", url)
        url = url.rstrip("/")

        # Maintain stack to find parent
        while stack and stack[-1][0] >= depth:
            stack.pop()

        parent_url = stack[-1][1] if stack else current_category

        adjacency.setdefault(parent_url, []).append(url)
        stack.append((depth, url))

    return adjacency

def find_leaves(adjacency):
    """Return URLs that never appear as a parent (leaf nodes)."""
    all_children = {child for children in adjacency.values() for child in children}
    all_parents = set(adjacency.keys())
    leaves = sorted(all_children - all_parents)
    return leaves

def main():
    print("Parsing RTF crawl log…")
    lines = extract_lines_from_rtf(INPUT_FILE)
    print(f"Found {len(lines)} lines.")

    adjacency = build_adjacency(lines)
    print(f"Built tree with {len(adjacency)} parent nodes.")

    leaves = find_leaves(adjacency)
    print(f"Identified {len(leaves)} leaf URLs (likely product pages).")

    # Save results
    with open(OUTPUT_TREE, "w", encoding="utf-8") as f:
        json.dump(adjacency, f, indent=2, ensure_ascii=False)

    with open(OUTPUT_LEAVES, "w", encoding="utf-8") as f:
        json.dump(leaves, f, indent=2, ensure_ascii=False)

    print(f"✅ Saved:\n - {OUTPUT_TREE}\n - {OUTPUT_LEAVES}")

if __name__ == "__main__":
    main()


Parsing RTF crawl log…
Found 594 lines.
Built tree with 8 parent nodes.
Identified 393 leaf URLs (likely product pages).
✅ Saved:
 - hobart_tree.json
 - hobart_leaf_urls.json
